# EcoVision · Week 3 — Independent Final Test Evaluation

**Project:** `ecovision-flutter-ml`  
**Candidate locked from validation:** frozen MobileNetV2, checkpoint epoch 8 (`mobilenetv2_frozen_20261103_155149`).  
**Test split:** 380 images, independent baseline v1 split.  
**Runtime:** VS Code Jupyter kernel **Python (EcoVision ML - WSL GPU)** (CPU fallback is okay).

## Important protocol

1. Run **Preflight (Cells A–D)** and inspect every assertion **before** accessing the test images for inference.
2. Verify that class order matches the *Week 2 notebook*, not just `labels.json`.
3. Confirm the preprocessing below matches the Week 2 training preprocessing (RGB decode, bilinear resize 224×224, float32 0–255, model-internal Rescaling).
4. **Only then** run the final test evaluation cells once for the preselected frozen baseline. Do not tune the model, reselect a checkpoint, or compare alternatives using the test metrics.
5. Outputs go to the Git-ignored `ml/reports/generated/week3/`. Report only actual results.

No test results have been computed in this provided notebook; run it on your own local data and checkpoint.

In [ ]:
# A. Imports and project paths — no test inference
from pathlib import Path
import ast
import hashlib
import json
import time

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, precision_recall_fscore_support,
)

EXPECTED_LABELS = ["cardboard", "glass", "metal", "paper", "plastic", "trash"]
EXPECTED_SHA256 = "0d560451fbd667f42c852752436a0ffc77f8068b8bd3b326ded4bd16bc3e6023"
EXPECTED_TEST_COUNTS = {
    "cardboard": 60, "glass": 75, "metal": 61,
    "paper": 89, "plastic": 74, "trash": 21,
}

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "ml/data/splits/v1/test.csv").exists() and (candidate / "ml/notebooks/02_mobilenetv2_baseline.ipynb").exists():
            return candidate.resolve()
    raise FileNotFoundError("Cannot locate EcoVision root from current Jupyter working directory")

ROOT = find_project_root()
SPLITS = ROOT / "ml/data/splits/v1"
DATA_ROOT = ROOT / "ml/data/raw/dataset-resized"
LABELS_PATH = ROOT / "ml/models/v1/labels.json"
TRAINING_NB = ROOT / "ml/notebooks/02_mobilenetv2_baseline.ipynb"
CHECKPOINT = ROOT / "ml/reports/generated/week2/mobilenetv2_frozen_20261103_155149/best_model.keras"
OUTPUT = ROOT / "ml/reports/generated/week3"

print("Repo root:", ROOT)
print("TensorFlow:", tf.__version__)
print("Detected GPUs:", tf.config.list_physical_devices("GPU"))
print("Data root exists:", DATA_ROOT.is_dir())
print("Frozen checkpoint exists:", CHECKPOINT.is_file())
assert DATA_ROOT.is_dir(), DATA_ROOT
assert CHECKPOINT.is_file(), CHECKPOINT
assert LABELS_PATH.is_file(), LABELS_PATH

### B. Verify actual training class mapping

This extracts `CLASS_NAMES` assignments from the *saved code cells* of the Week 2 notebook and checks that every recoverable assignment agrees with `labels.json`. An inability to recover the mapping is a **stop condition**; do not assume alphabetical order is enough.

In [ ]:
labels = json.loads(LABELS_PATH.read_text(encoding="utf-8"))
assert labels == EXPECTED_LABELS, f"labels.json unexpected: {labels}"

week2_notebook = json.loads(TRAINING_NB.read_text(encoding="utf-8"))
recovered = []
for cell_index, cell in enumerate(week2_notebook["cells"]):
    if cell.get("cell_type") != "code":
        continue
    source = cell.get("source", "")
    if isinstance(source, list):
        source = "".join(source)
    try:
        tree = ast.parse(source)
    except SyntaxError:
        continue
    for node in ast.walk(tree):
        value_node = None
        if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == "CLASS_NAMES" for t in node.targets):
            value_node = node.value
        elif isinstance(node, ast.AnnAssign) and isinstance(node.target, ast.Name) and node.target.id == "CLASS_NAMES":
            value_node = node.value
        if value_node is not None:
            try:
                value = ast.literal_eval(value_node)
            except (ValueError, TypeError):
                continue
            recovered.append((cell_index, list(value)))

print("labels.json:", labels)
print("CLASS_NAMES found in Week 2 notebook:")
for cell_index, value in recovered:
    print(f"  code cell {cell_index}: {value}")
assert recovered, "Could not confirm Week 2 CLASS_NAMES; inspect notebook manually before test evaluation"
assert all(value == labels for _, value in recovered), "Training CLASS_NAMES disagrees with labels.json"
LABEL_TO_INDEX = {name: i for i, name in enumerate(labels)}
print("PASS: model label mapping is consistent with saved Week 2 code")

### C. Verify split manifests, membership and source files

This reads the split CSVs and **hashes all 380 test images** to ensure that the local test files match the frozen v1 manifest. It also checks that no SHA or group ID crosses splits. This is data integrity checking, **not model selection or test inference**.

In [ ]:
def file_sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

frames = {}
for split in ("train", "val", "test"):
    frame = pd.read_csv(SPLITS / f"{split}.csv", dtype=str)
    required = {"path", "label", "sha256", "group_id", "split"}
    assert required.issubset(frame.columns), f"Missing columns in {split}"
    assert frame["split"].eq(split).all(), f"Incorrect split column: {split}"
    assert frame["label"].isin(labels).all(), f"Unexpected class in {split}"
    assert frame[["path", "label", "sha256", "group_id", "split"]].notna().all().all()
    frames[split] = frame

assert len(frames["train"]) == 1766
assert len(frames["val"]) == 378
assert len(frames["test"]) == 380

for key in ("path", "sha256", "group_id"):
    if key == "group_id":
        # Groups repeat inside a split by design, but never across different splits.
        pass
    for left, right in (("train", "val"), ("train", "test"), ("val", "test")):
        common = set(frames[left][key]) & set(frames[right][key])
        assert not common, f"{key} overlaps {left}/{right}: {list(common)[:3]}"

test_df = frames["test"].copy().reset_index(drop=True)
observed = test_df["label"].value_counts().to_dict()
assert observed == EXPECTED_TEST_COUNTS, f"Unexpected test class counts: {observed}"

for i, row in test_df.iterrows():
    relative = Path(row["path"])
    assert not relative.is_absolute() and ".." not in relative.parts, f"Unsafe path: {relative}"
    image_file = DATA_ROOT / relative
    assert image_file.is_file(), f"Missing image: {image_file}"
    assert file_sha256(image_file) == row["sha256"], f"SHA-256 mismatch: {relative}"

print("PASS: expected split sizes and six-class test distribution")
print("PASS: no cross-split path, SHA256, or group_id overlap")
print("PASS: all 380 local test files match SHA256 manifest")
print("Test distribution:", observed)

### D. Verify frozen model, checksum, preprocessing contract

The batch input stays **RGB float32 0–255**; `Rescaling(scale=1/127.5, offset=-1)` is inside the Keras model. No external MobileNetV2 preprocessing should be applied.

In [ ]:
assert file_sha256(CHECKPOINT) == EXPECTED_SHA256, "Frozen checkpoint differs from verified Week 2 backup"
model = tf.keras.models.load_model(CHECKPOINT, compile=False)
assert tuple(model.input_shape[1:]) == (224, 224, 3), model.input_shape
assert tuple(model.output_shape[1:]) == (6,), model.output_shape
assert model.inputs[0].dtype == "float32"
assert model.outputs[0].dtype == "float32"
rescaling = [layer for layer in model.layers if isinstance(layer, tf.keras.layers.Rescaling)]
assert len(rescaling) == 1, f"Expected one model-internal Rescaling layer, got {len(rescaling)}"
assert np.isclose(rescaling[0].scale, 1 / 127.5)
assert np.isclose(rescaling[0].offset, -1.0)

print("PASS: checkpoint SHA-256 is exactly the Week 2 frozen epoch-8 checkpoint")
print("PASS: model contract input float32 [batch,224,224,3] -> output float32 [batch,6]")
print("PASS: model-internal rescaling to [-1,1]")
print("PRE-FLIGHT COMPLETE — confirm matching Week 2 image resize/decode approach before evaluating the test set")

---

## Final test evaluation — run only after all preflight assertions pass

**Decision frozen before viewing test metrics:** evaluate **only** `mobilenetv2_frozen_20261103_155149/best_model.keras`. The fine-tuned checkpoint is not compared on the test split for selection.

**Confirm preprocessing against your Week 2 `02_mobilenetv2_baseline.ipynb`:** RGB decode to 3 channels, resize with TensorFlow default **bilinear** to 224×224, float32 pixel values in [0,255], **no external normalization**. If Week 2 used a different resize, decoding, or color conversion method, align the following loader *before running*, not after inspecting results.

Running the next cell performs the independent final test-set inference. Keep test results for final reporting; do not iterate to improve test metrics.

In [ ]:
# E. Locked final test inference. Read this cell before executing.
# Run only once the preflight and preprocessing contract above are confirmed.


def load_rgb_image(relative_path):
    path = tf.strings.join([
        str(DATA_ROOT) + "/",
        relative_path
    ])

    image_bytes = tf.io.read_file(path)

    image = tf.io.decode_image(
        image_bytes,
        channels=3,
        expand_animations=False
    )

    image.set_shape([None, None, 3])

    image = tf.image.resize(
        image,
        (224, 224),
        method="bilinear",
        antialias=True
    )

    image = tf.ensure_shape(image, [224, 224, 3])

    return tf.cast(image, tf.float32)


BATCH_SIZE = 8
image_paths = test_df["path"].astype(str).tolist()
true_indices = np.array([LABEL_TO_INDEX[label] for label in test_df["label"]], dtype=np.int64)

dataset = (
    tf.data.Dataset.from_tensor_slices(image_paths)
    .map(load_rgb_image, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

start = time.perf_counter()
probabilities = model.predict(dataset, verbose=1)
elapsed = time.perf_counter() - start

assert probabilities.shape == (380, 6), probabilities.shape
assert np.isfinite(probabilities).all(), "Non-finite probabilities"
assert np.allclose(probabilities.sum(axis=1), 1.0, atol=1e-4), "Output probability sums invalid"
pred_indices = np.argmax(probabilities, axis=1)

print("PASS: final frozen baseline inference completed for 380 test images")
print("Elapsed wall time (includes pipeline/setup; NOT Android inference latency):", round(elapsed, 2), "seconds")

## Final test metrics and artifacts

**Required metrics:** accuracy, macro precision, macro recall, macro-F1, per-class precision/recall/F1/support, confusion matrix. The plot and results are generated from actual output; nothing is hardcoded. Files under `ml/reports/generated/` are kept out of Git.

In [ ]:
# F. Final metrics from frozen test predictions
labels_idx = list(range(len(labels)))
accuracy = accuracy_score(true_indices, pred_indices)
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    true_indices, pred_indices, labels=labels_idx, average="macro", zero_division=0
)
report = classification_report(
    true_indices, pred_indices, labels=labels_idx,
    target_names=labels, output_dict=True, zero_division=0
)
cm = confusion_matrix(true_indices, pred_indices, labels=labels_idx)

print("=== ECOVISION WEEK 3 — FROZEN BASELINE FINAL TEST (n=380) ===")
print(f"Accuracy:         {accuracy:.4%}")
print(f"Macro precision:  {macro_precision:.4%}")
print(f"Macro recall:     {macro_recall:.4%}")
print(f"Macro F1:         {macro_f1:.4%}")
print("Correct samples: ", int((true_indices == pred_indices).sum()), "/ 380")

class_metrics = pd.DataFrame(report).T.loc[labels, ["precision", "recall", "f1-score", "support"]]
display(class_metrics)
print("Confusion matrix rows=true, columns=predicted:\n", cm)

In [ ]:
# G. Plot confusion matrix (absolute counts)
OUTPUT.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(9, 7))
display_cm = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
display_cm.plot(ax=ax, xticks_rotation=45, values_format="d", colorbar=False)
ax.set_title("EcoVision · Frozen MobileNetV2 · Final test (n=380)")
fig.tight_layout()
plot_path = OUTPUT / "frozen_final_test_confusion_matrix.png"
fig.savefig(plot_path, dpi=160, bbox_inches="tight")
plt.show()
print("Saved:", plot_path)

In [ ]:
# H. Save reproducible, git-ignored run outputs (no raw dataset images)
metrics = {
    "model_id": "mobilenetv2_frozen_20261103_155149",
    "checkpoint_sha256": EXPECTED_SHA256,
    "split_version": "v1",
    "label_order": labels,
    "test_samples": len(test_df),
    "accuracy": float(accuracy),
    "macro_precision": float(macro_precision),
    "macro_recall": float(macro_recall),
    "macro_f1": float(macro_f1),
    "confusion_matrix": cm.tolist(),
    "per_class": {
        label: {k: float(report[label][k]) if k != "support" else int(report[label][k])
                for k in ("precision", "recall", "f1-score", "support")}
        for label in labels
    },
    "notes": [
        "Checkpoint chosen from validation before test evaluation",
        "Only independent TrashNet v1 test set evaluated",
        "Elapsed notebook wall time is not Android device inference latency",
        "Real-world smartphone testing not yet performed",
    ],
}

(OUTPUT / "frozen_final_test_metrics.json").write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
class_metrics.to_csv(OUTPUT / "frozen_final_test_per_class.csv", index_label="class")

predictions_df = test_df[["path", "label", "sha256"]].copy()
predictions_df["predicted_label"] = [labels[i] for i in pred_indices]
predictions_df["confidence"] = probabilities.max(axis=1)
predictions_df["correct"] = true_indices == pred_indices
predictions_df.to_csv(OUTPUT / "frozen_final_test_predictions.csv", index=False)

print("Saved in:", OUTPUT)
for name in ("frozen_final_test_metrics.json", "frozen_final_test_per_class.csv", "frozen_final_test_predictions.csv", "frozen_final_test_confusion_matrix.png"):
    print(" -", name)
print("Do not change model or thresholds based on these test results")

## Interpretation (complete only after observed results)

- Compare **frozen validation** accuracy 85.98%, macro-F1 82.73%, and trash recall 57.14% with actual final test scores, while noting split sampling uncertainty.
- Examine low-support class `trash` (test support 21) separately; use per-class recall/F1, not only overall accuracy.
- Discuss errors and domain shift due to controlled TrashNet backgrounds.
- No test-based hyperparameter tuning; use future independent smartphone photos as a **separate** real-world evaluation.
- The Week 3 TFLite export and TensorFlow↔TFLite parity test are separate follow-up steps.

**Before Git commit:** write a tracked, hand-reviewed `ml/reports/week3_final_test_evaluation.md` using the actual metrics. Keep generated bulk artifacts in the ignored reports directory.